In [1]:
import streamlit as st
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from mlxtend.frequent_patterns import apriori, association_rules

import matplotlib.pyplot as plt

In [2]:
st.set_page_config(page_title='Book Recommendation System', layout='wide')

st.title('Book Recommendation System')


def load_data():
    books = pd.read_csv('books_clean.csv')
    trending = pd.read_csv('trending_clean.csv')
    return books, trending


books, trending = load_data()

def build_content_model(df):
    tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = tfidf.fit_transform(df['content'])

    sim = cosine_similarity(tfidf_matrix)

    book_index = pd.Series(df.index, index=df['book_title']).drop_duplicates()

    return sim, book_index, df


content_sim, book_idx, df_cb = build_content_model(books)


def recommend_content(title, top_n=10):

    if title not in book_idx:
        return pd.DataFrame()

    idx = book_idx[title]

    scores = list(enumerate(content_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    indices = [i[0] for i in scores]

    recs = df_cb.iloc[indices][['book_title', 'rating']]
    recs['similarity'] = [i[1] for i in scores]

    return recs


def build_basket(df):

    basket = df.groupby(['user_id', 'book_title'])['rating'].count().unstack().fillna(0)
    basket = basket.applymap(lambda x: 1 if x > 0 else 0)

    return basket


basket = build_basket(books)


def run_apriori(basket):

    freq = apriori(basket, min_support=0.002, use_colnames=True)

    rules = association_rules(freq, metric='lift', min_threshold=1)

    return rules


rules = run_apriori(basket)


menu = st.sidebar.radio(
    'Select View',
    [
        'Content-Based Recommendations',
        'Market Basket Analysis',
        'Trending Books',
        'Analytics Dashboard'
    ]
)



if menu == 'Content-Based Recommendations':

    st.header('Content-Based Book Recommendations')

    book_name = st.text_input('Enter book title')

    if book_name:

        recs = recommend_content(book_name, top_n=10)

        if recs.empty:
            st.write('Book not found in dataset')
        else:
            st.dataframe(recs, use_container_width=True)


if menu == 'Market Basket Analysis':

    st.header('Books Frequently Bought Together')

    min_lift = st.slider('Minimum Lift', 1.0, 20.0, 5.0)

    filtered_rules = rules[
        (rules['lift'] >= min_lift) &
        (rules['confidence'] >= 0.3)
    ]

    st.dataframe(
        filtered_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20),
        use_container_width=True
    )


if menu == 'Trending Books':

    st.header('Top 100 Trending Books')
    st.dataframe(trending, use_container_width=True)


if menu == 'Analytics Dashboard':

    st.header('Dataset Insights from Top 100 Trending Books')

    col1, col2, col3 = st.columns(3)

    # ---------------- POPULAR BOOKS ----------------
    top_books = trending['book title'].value_counts().head(10)

    fig1, ax1 = plt.subplots()
    top_books.plot(kind='bar', ax=ax1)
    ax1.set_title('Most Popular Trending Books')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)

    col1.pyplot(fig1)


    # ---------------- GENRES ----------------
    genre_counts = trending['genre'].value_counts().head(10)

    fig2, ax2 = plt.subplots()
    genre_counts.plot(kind='bar', ax=ax2)
    ax2.set_title('Top Genres in Trending Books')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=45)

    col2.pyplot(fig2)


    # ---------------- AUTHORS ----------------
    author_counts = trending['author'].value_counts().head(10)

    fig3, ax3 = plt.subplots()
    author_counts.plot(kind='bar', ax=ax3)
    ax3.set_title('Top Authors in Trending Books')
    ax3.set_ylabel('Count')
    ax3.tick_params(axis='x', rotation=45)

    col3.pyplot(fig3)


2026-05-23 19:58:47.089 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-23 19:58:47.090 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-23 19:58:47.647 
  command:

    streamlit run C:\Users\Blaithin\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-23 19:58:47.648 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-23 19:58:47.649 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


KeyboardInterrupt: 